# Aqua Epoch — Phase 2: honest validation
Train/evaluate full and quick classifiers plus pH regressors without modifying Phase 1/backend artifacts. CPU runtime is sufficient; no GPU or API key is needed.

**Colab setup:** upload the entire project folder (not just this notebook/CSV) to `MyDrive/Aqua_Epoch`, including `ml/`, `backend/`, and requirements files. Open this notebook in Colab. If your folder differs, edit `PROJECT_ROOT` below. Drive access will ask for your approval.
**Local Jupyter:** open this notebook from this project's `notebooks/` directory or the project root.
**Important:** the dataset's units/provenance are undocumented. Internal validation is not proof of real-world drinking-water safety. This notebook does not use LLM APIs.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
from datetime import datetime, timezone
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Aqua_Epoch')
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ml' / 'evaluate.py').exists()), None)
if PROJECT_ROOT is None or not (PROJECT_ROOT / 'ml' / 'evaluate.py').is_file():
    raise FileNotFoundError('Upload the complete project, then set PROJECT_ROOT to the folder containing ml/evaluate.py.')
sys.path.insert(0, str(PROJECT_ROOT))
DATASET = PROJECT_ROOT / 'Aqua_Epoch_BIS_Water_Quality_Dataset.xls - Aqua_Epoch_BIS_Water_Quality_Dataset.csv'
assert DATASET.is_file(), f'Dataset not found: {DATASET}'
# Default is a real training run. Set False to inspect existing validated evidence.
RUN_EXPERIMENTS = os.getenv('AQUA_RUN_EXPERIMENTS', '1') == '1'
INSTALL_DEPENDENCIES = os.getenv('AQUA_SKIP_INSTALL', '0') != '1'
RESULTS = PROJECT_ROOT / 'reports' / 'phase2_validated'
if RUN_EXPERIMENTS:
    RESULTS = PROJECT_ROOT / 'reports' / ('phase2_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
print('Project:', PROJECT_ROOT, '\nOutput:', RESULTS)

In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')], cwd=PROJECT_ROOT, check=True)
# If Colab reports that a loaded package was upgraded, restart runtime and rerun.

## Protocol before looking at scores
- Preserve source row IDs and audit blank/unlabelled/invalid measurements. Never invent target values.
- Compare raw and physically sanitized features. Positive spikes are not automatically deleted.
- Fit preprocessing only inside training folds. Four candidates, three-fold training CV, 20% internal holdout.
- Select classifiers using training CV F1 and regressors using training CV RMSE. Report both modes/policies rather than choosing using holdout scores.
- Safe is class 1. Unsafe-to-safe FPR is the dangerous error. Choose a threshold >=0.5 using training OOF predictions only, targeting empirical FPR <=2%. This target is not guaranteed on new data.
- Source-order and seed-7/91 splits are sensitivity checks, not independent external tests.
- Raw-pH and [0,14]-target regression cohorts are different populations. Show both, including a mean baseline; don't advertise their difference as pure model improvement.

In [ ]:
from ml.quality import load_quality_data
frame, audit = load_quality_data(DATASET)
print('Raw:', audit['rows_read'], 'Labelled:', len(frame))
print('Blank:', audit['fully_blank_rows_removed'], 'Nonblank unlabelled:', audit['missing_target_nonblank_rows_removed'])
print('Invalid observed pH:', audit['labelled']['invalid_ph_observed'])
print('Negative hardness + fluoride on same rows:', audit['labelled']['paired_negative_hardness_and_fluoride'])
for name, view in audit['statistical_views'].items():
    print(name, 'TDS/Turbidity correlation:', view['pearson_correlations']['TDS']['Turbidity'])

In [ ]:
if RUN_EXPERIMENTS:
    subprocess.run([sys.executable, '-m', 'ml.evaluate', '--dataset', str(DATASET), '--output', str(RESULTS), '--folds', '3', '--trees', '80', '--permutation-repeats', '3'], cwd=PROJECT_ROOT, check=True)
else:
    assert (RESULTS / 'results.json').is_file(), 'No existing results. Set RUN_EXPERIMENTS=True.'

In [ ]:
from ml.report import generate_report
from IPython.display import display, Markdown, Image
summary = generate_report(RESULTS)
print('Completed experiments:', len(summary['experiments']))
print('Evidence directory:', RESULTS)
for e in summary['experiments']:
    print(e['id'], e['model'], 'CV:', e['cv'], '\nHoldout:', e['holdout'])

In [ ]:
for figure in summary['figures']:
    display(Image(filename=str(RESULTS / figure)))

## Experimental inference — not automatic promotion
The existing backend still serves Phase 1. Here we load a candidate **we generated ourselves**, use its recorded column order and safety threshold, and make one prediction. Never load joblib files from untrusted uploads. The example uses a heldout row so it isn't a training-row demonstration; it is not an unbiased evaluation by itself. Full pH regression does not receive pH or Potability.

In [ ]:
import joblib, numpy as np
name = 'classification_binary_targets_full_physical_sanitized'
bundle = joblib.load(RESULTS / 'candidates' / (name + '.joblib'))
metadata, pipeline = bundle['metadata'], bundle['pipeline']
row_id = metadata['holdout_row_ids'][0]
X = frame.loc[[row_id], metadata['feature_order']]
p_safe = float(pipeline.predict_proba(X)[0, list(pipeline.classes_).index(1)])
threshold = metadata['threshold']['value']
print({'csv_line': row_id + 2, 'probability_safe': p_safe, 'threshold': threshold, 'predicted_class': int(p_safe >= threshold), 'experimental': True})
reg = joblib.load(RESULTS / 'candidates' / 'regression_physical_valid_targets_full_physical_sanitized.joblib')
reg_id = reg['metadata']['holdout_row_ids'][0]
assert 'pH' not in reg['metadata']['feature_order'] and 'Potability' not in reg['metadata']['feature_order']
print('Estimated pH (not lab measurement):', float(reg['pipeline'].predict(frame.loc[[reg_id], reg['metadata']['feature_order']])[0]))

## Before deployment
Read `REVIEW.md`, check false-safe errors, safe precision/recall/coverage, regression skill vs dummy, and source-order stress tests. Full classification's high score only describes these labels; source provenance and near-duplicate grouping are still unverified.
Phase 3 must integrate a chosen policy, explicit uncertainty/out-of-distribution warnings and correct DataFrame/threshold inference into the backend. No Phase 2 candidate is automatically approved. UI and PDF remain deferred.
Outputs are saved in RESULTS (on Drive in Colab). The notebook is run locally with the same Python modules; Colab execution itself must still be checked in your account.